# GCP course notebook (CPU kernel)

This JupyterLab kernel is **CPU**. TPU work runs on the shared GKE v5e pool via `submit_tpu`.

This is the GCP path only: there is no Colab runtime, no Kaggle notebook kernel, and no PSC node. Dataset paths stay on this notebook disk or on the TPU Job (which cannot see this PVC).

- Run **Setup** here to confirm JAX sees CPU.
- Do **not** train or call `jax.devices()` expecting a TPU in this kernel.
- Run **Submit**: `submit_tpu.smoke()` for a 15s chip check, or `submit_tpu.submit()` to assemble the marked cells below and run them on v5e.
- Packages (JAX, Flax, pillow, …) are already in the course image. Do not pip-install at runtime.


# Setup


In [ ]:
USE_TPU = False  # True only inside the GKE TPU Job (submit_tpu sets it)


In [ ]:
# Course image already has JAX, Flax, pillow, augmax, wandb, torchvision.
# No pip on this CPU kernel and none on the TPU Job.


In [ ]:
import os
from pathlib import Path

# Pin CPU *before* import jax. The hub image ships libtpu; leftover TPU_* env
# vars otherwise make import crash (TPU_ACCELERATOR_TYPE / TPU_WORKER_HOSTNAMES).
# submit_tpu sets TPU_JOB=True and JAX_PLATFORMS=tpu in the Job container.
if not globals().get("TPU_JOB", False):
    os.environ["JAX_PLATFORMS"] = "cpu"
    os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

print("cwd:", Path.cwd())
print("JAX_PLATFORMS:", os.environ.get("JAX_PLATFORMS"))

import jax
print("JAX devices:", jax.devices())


In [ ]:
import jax
import jax.numpy as jnp

DEVICE = jax.devices()[0]
print("Device:", DEVICE)


# TPU work

The next two cells are what `submit_tpu.submit()` sends to the chip. They no-op in this kernel.


In [ ]:
def tpu_main():
    """Runs only on the v5e Job, not in this CPU kernel."""
    import jax
    import jax.numpy as jnp

    devices = jax.devices()
    print("devices:", devices, flush=True)
    print("platform:", devices[0].platform, flush=True)
    assert devices[0].platform == "tpu", devices
    x = jnp.ones((1024, 1024), dtype=jnp.bfloat16)
    y = (x @ x).block_until_ready()
    print("matmul checksum:", float(y.sum()), flush=True)


In [ ]:
if not globals().get("TPU_JOB", False):
    print("Skipping in-kernel TPU work. Use the Submit cell (submit_tpu.submit or smoke).")
else:
    tpu_main()


# Submit to TPU

The v5e pod **cannot mount this Jupyter PVC**. `submit_tpu.submit()` concatenates the marked Setup/TPU cells, runs them on one chip, and streams logs into the next cell.

Interrupt that cell to cancel the Job.


In [ ]:
import sys
from pathlib import Path

# Hub-mounted helper wins over a stale copy in the home PVC.
_course = Path("/opt/course")
if _course.exists():
    sys.path.insert(0, str(_course))

import submit_tpu

# Chip check only (fast). Swap to submit() to run tpu_main() on v5e.
submit_tpu.smoke()
# submit_tpu.submit()
